In [33]:
!pip install yfinance

In [34]:
import random
import joblib
import yfinance as yf
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, mean_absolute_error

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


In [35]:

# --- CELL 3 - Configurable Data Parameters ---
TICKER = TICKERS = [
    "NVDA", "AMZN", "AXON", "AAPL", "ORCL",
    "MSFT", "JPM",  "META", "TSLA", "AMD"
]
MAX_DOWNLOAD_PERIOD = "5y"
PERIOD_CANDIDATES = ["1y", "2y", "3y", "4y", "5y"]

# Download one maximum history window, then tune the best period dynamically
df_full = yf.download(TICKER, period=MAX_DOWNLOAD_PERIOD, interval="1d", progress=False, auto_adjust=True)

if isinstance(df_full.columns, pd.MultiIndex):
    df_full.columns = [str(c[0]) for c in df_full.columns]

# Force deduplicate — yfinance sometimes returns two "Close" columns
df_full = df_full.loc[:, ~df_full.columns.duplicated()]
df_full = df_full.reset_index().sort_values("Date").reset_index(drop=True)
print(f"✅ Downloaded {len(df_full)} days of historical data for {TICKER}.")
print(f"Dynamic period candidates: {PERIOD_CANDIDATES}")


spy_series = yf.download("SPY", period=MAX_DOWNLOAD_PERIOD, interval="1d",
                         progress=False, auto_adjust=True)["Close"].pct_change()
vix_series = yf.download("^VIX", period=MAX_DOWNLOAD_PERIOD, interval="1d",
                         progress=False, auto_adjust=True)["Close"]

if isinstance(spy_series, pd.DataFrame): spy_series = spy_series.iloc[:, 0]
if isinstance(vix_series, pd.DataFrame): vix_series = vix_series.iloc[:, 0]
spy_series.index = pd.to_datetime(spy_series.index)
vix_series.index = pd.to_datetime(vix_series.index)
spy_series.name  = "spy_return"
vix_series.name  = "vix_level"

df_full = df_full.set_index("Date")
df_full = df_full.join(spy_series, how="left").join(vix_series, how="left")
df_full = df_full.reset_index()
df_full[["spy_return", "vix_level"]] = df_full[["spy_return", "vix_level"]].ffill()

print(f"df_full shape after SPY/VIX merge: {df_full.shape}")
print(f"NaN check — spy: {df_full['spy_return'].isna().sum()}, vix: {df_full['vix_level'].isna().sum()}")


✅ Downloaded 1256 days of historical data for ['NVDA', 'AMZN', 'AXON', 'AAPL', 'ORCL', 'MSFT', 'JPM', 'META', 'TSLA', 'AMD'].
Dynamic period candidates: ['1y', '2y', '3y', '4y', '5y']
df_full shape after SPY/VIX merge: (1256, 8)
NaN check — spy: 1, vix: 0


In [36]:
# ══════════════════════════════════════════════════════════════
# CELL 4 — Feature Engineering (updated for 1y–5y periods)
# ══════════════════════════════════════════════════════════════

def engineer_features(raw_df: pd.DataFrame) -> pd.DataFrame:

    # ── NUCLEAR FLATTEN ───────────────────────────────────────
    flat = {}
    src = raw_df.copy()
    if isinstance(src.columns, pd.MultiIndex):
        src.columns = [str(c[0]) if isinstance(c, tuple) else str(c) for c in src.columns]

    for col in ["Open", "High", "Low", "Close", "Volume"]:
        if col in src.columns:
            extracted = src[col]
            if isinstance(extracted, pd.DataFrame):
                flat[col] = extracted.iloc[:, 0].to_numpy(dtype=float, na_value=np.nan)
            else:
                flat[col] = extracted.to_numpy(dtype=float, na_value=np.nan)

    if "Date" in src.columns:
        flat["Date"] = src["Date"].values

    for extra_col in ["spy_return", "vix_level"]:
        if extra_col in src.columns:
            extracted = src[extra_col]
            if isinstance(extracted, pd.DataFrame):
                flat[extra_col] = extracted.iloc[:, 0].to_numpy(dtype=float, na_value=np.nan)
            else:
                flat[extra_col] = extracted.to_numpy(dtype=float, na_value=np.nan)

    df = pd.DataFrame(flat)

    # ── Features ──────────────────────────────────────────────
    close  = df["Close"]
    volume = df["Volume"]

    # Short-term MAs (unchanged)
    df["sma_10"] = close.rolling(10).mean()
    df["sma_20"] = close.rolling(20).mean()

    # ✅ CHANGE 1: Add longer-term MAs — meaningful only with 2y+ data
    # With 5y data, 50-day and 200-day MAs have enough history to be reliable [web:86]
    df["sma_50"]  = close.rolling(50).mean()
    df["sma_200"] = close.rolling(200).mean()

    # RSI (unchanged)
    delta    = close.diff()
    gain     = delta.clip(lower=0)
    loss     = -delta.clip(upper=0)
    avg_gain = gain.rolling(14).mean()
    avg_loss = loss.rolling(14).mean()
    rs       = avg_gain / avg_loss.replace(0, np.nan)
    df["rsi_14"] = 100 - (100 / (1 + rs))

    # MACD (unchanged)
    ema12 = close.ewm(span=12, adjust=False).mean()
    ema26 = close.ewm(span=26, adjust=False).mean()
    df["macd"]        = ema12 - ema26
    df["macd_signal"] = df["macd"].ewm(span=9, adjust=False).mean()

    # Bollinger Bands (unchanged)
    rolling_mean  = close.rolling(20).mean()
    rolling_std   = close.rolling(20).std()
    df["bb_upper"] = rolling_mean + (2 * rolling_std)
    df["bb_lower"] = rolling_mean - (2 * rolling_std)
    df["bb_width"] = (df["bb_upper"] - df["bb_lower"]) / close

    # Returns & Volatility (unchanged)
    df["return"]         = close.pct_change()
    df["return_3d"]      = close.pct_change(3)
    df["return_5d"]      = close.pct_change(5)
    df["volatility_5d"]  = df["return"].rolling(5).std()
    df["volatility_10d"] = df["return"].rolling(10).std()

    # ✅ CHANGE 2: Add longer-term volatility — captures regime shifts across 3y–5y data [web:92]
    df["volatility_20d"] = df["return"].rolling(20).std()

    # Distance from MAs (short + new long-term)
    df["dist_sma_10"]  = (close - df["sma_10"])  / df["sma_10"]
    df["dist_sma_20"]  = (close - df["sma_20"])  / df["sma_20"]

    # ✅ CHANGE 3: Add distance from long-term MAs — critical for 3y–5y [web:86][web:92]
    df["dist_sma_50"]  = (close - df["sma_50"])  / df["sma_50"]
    df["dist_sma_200"] = (close - df["sma_200"]) / df["sma_200"]

    # ✅ CHANGE 4: Add 52-week high/low distance — only useful with 1y+ data [web:84]
    rolling_52w_high   = close.rolling(252).max()
    rolling_52w_low    = close.rolling(252).min()
    df["dist_52w_high"] = (close - rolling_52w_high) / rolling_52w_high
    df["dist_52w_low"]  = (close - rolling_52w_low)  / rolling_52w_low

    # ✅ CHANGE 5: Volume-based features — more stable with 2y+ data [web:83]
    df["volume_sma_20"]  = volume.rolling(20).mean()
    df["volume_ratio"]   = volume / df["volume_sma_20"].replace(0, np.nan)

    # Targets
    # ✅ CHANGE 6: Use 0.001 threshold (>0.1% move = BUY) — fixes class imbalance
    df["target_signal"] = (close.shift(-1) / close - 1 > 0.001).astype(int)
    df["target_price"]  = (close.shift(-1) - close) / close    # exact next-day % return

    df = df.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)
    return df


# ── Period mapping (unchanged — keep all rows defined even if not all used) ──
PERIOD_TO_ROWS = {
    "6mo":  126,
    "9mo":  189,
    "1y":   252,
    "18mo": 378,
    "2y":   504,
    "3y":   756,
    "4y":  1008,
    "5y":  1260,
}


def get_period_df(raw_df: pd.DataFrame, period_label: str) -> pd.DataFrame:
    n      = PERIOD_TO_ROWS[period_label]
    sliced = raw_df.tail(min(len(raw_df), n)).copy()
    return engineer_features(sliced)


# ✅ CHANGE 7: Use only 1y–5y periods
PERIOD_CANDIDATES = ["1y", "2y", "3y", "4y", "5y"]

period_featured_dfs = {}
for p in PERIOD_CANDIDATES:
    df_tmp = get_period_df(df_full, p)
    if len(df_tmp) >= 120:
        period_featured_dfs[p] = df_tmp

if not period_featured_dfs:
    raise ValueError("Not enough data after feature engineering for any candidate period.")

print("Usable periods:", {k: len(v) for k, v in period_featured_dfs.items()})

# ✅ Class balance diagnostic
print("\nClass balance check:")
for period, df_p in period_featured_dfs.items():
    buy_pct  = df_p["target_signal"].mean() * 100
    sell_pct = 100 - buy_pct
    status   = "✅" if 40 <= buy_pct <= 65 else "⚠️ IMBALANCED"
    print(f"  {period:<5}: {len(df_p):>4} rows | BUY={buy_pct:.1f}% | SELL={sell_pct:.1f}%  {status}")

Usable periods: {'2y': 252, '3y': 504, '4y': 756, '5y': 1004}

Class balance check:
  2y   :  252 rows | BUY=49.2% | SELL=50.8%  ✅
  3y   :  504 rows | BUY=52.0% | SELL=48.0%  ✅
  4y   :  756 rows | BUY=50.4% | SELL=49.6%  ✅
  5y   : 1004 rows | BUY=50.2% | SELL=49.8%  ✅


In [37]:
# ══════════════════════════════════════════════════════════════
# CELL 5 — Feature Columns (updated for 1y–5y periods)
# ══════════════════════════════════════════════════════════════

feature_cols = [
    # ── Momentum ──────────────────────────────────────────────
    "rsi_14",              # overbought/oversold (works on all timeframes) [web:99]
    "macd",                # trend + momentum crossover signal [web:99]
    "macd_signal",         # MACD signal line for crossover detection

    # ── Volatility ────────────────────────────────────────────
    "bb_width",            # Bollinger band squeeze/expansion [web:102]
    "volatility_5d",       # short-term vol
    "volatility_10d",      # medium-term vol
    "volatility_20d",      # ✅ NEW — monthly regime volatility (needs 2y+ to be stable) [web:92]

    # ── Returns ───────────────────────────────────────────────
    "return_3d",           # 3-day momentum
    "return_5d",           # 5-day momentum

    # ── Distance from MAs ─────────────────────────────────────
    "dist_sma_10",         # short-term trend deviation
    "dist_sma_20",         # medium-term trend deviation
    "dist_sma_50",         # ✅ NEW — intermediate trend (50-day MA is most important) [web:103]
    "dist_sma_200",        # ✅ NEW — long-term trend, golden/death cross signal [web:103]

    # ── 52-Week High/Low ──────────────────────────────────────
    "dist_52w_high",       # ✅ NEW — support/resistance from yearly high [web:102]
    "dist_52w_low",        # ✅ NEW — support/resistance from yearly low  [web:102]

    # ── Volume ────────────────────────────────────────────────
    "volume_ratio",        # ✅ REPLACED raw Volume → ratio is scale-invariant [web:99]

    # ── Macro ─────────────────────────────────────────────────
    "spy_return",          # S&P 500 daily return (market context)
    "vix_level",           # CBOE VIX fear index [web:103]
]

print(f"Total features: {len(feature_cols)}")
print(f"Features: {feature_cols}")

Total features: 18
Features: ['rsi_14', 'macd', 'macd_signal', 'bb_width', 'volatility_5d', 'volatility_10d', 'volatility_20d', 'return_3d', 'return_5d', 'dist_sma_10', 'dist_sma_20', 'dist_sma_50', 'dist_sma_200', 'dist_52w_high', 'dist_52w_low', 'volume_ratio', 'spy_return', 'vix_level']


In [38]:
# ══════════════════════════════════════════════════════════════
# CELL 6 — Sequence Preparation (updated for 1y–5y periods)
# ══════════════════════════════════════════════════════════════

def create_sequences(X, y_sig, y_prc, lookback):
    Xs, ys_sig, ys_prc = [], [], []
    for i in range(lookback, len(X)):
        Xs.append(X[i - lookback:i])
        ys_sig.append(y_sig[i])
        ys_prc.append(y_prc[i])
    return np.array(Xs), np.array(ys_sig), np.array(ys_prc)


TEST_FRACTION = 0.20
VAL_FRACTION_WITHIN_TRAIN = 0.15

# ✅ CHANGE 1: Minimum test rows increased from 20 → 30
# With 1y–5y data the test set is larger; requiring 30 ensures meaningful evaluation [web:117]
MIN_TEST_ROWS = 30

# ✅ CHANGE 2: Minimum val rows increased from 20 → 25
MIN_VAL_ROWS = 25


def prepare_period_arrays(df_period, lookback):
    X        = df_period[feature_cols].values
    y_signal = df_period["target_signal"].values
    y_price  = df_period["target_price"].values

    # ── Chronological split — NO shuffling ever [web:113][web:115] ──
    split_index = int(len(df_period) * (1 - TEST_FRACTION))
    train_idx   = split_index

    # ✅ CHANGE 3: Fit scaler ONLY on training rows — prevents leakage
    # Critical with 5y data: if market regime changed, test scaling must
    # use only past (train) stats, never future (test) stats [web:113]
    feat_scaler  = StandardScaler()
    price_scaler = StandardScaler()
    feat_scaler.fit(X[:train_idx])
    price_scaler.fit(y_price[:train_idx].reshape(-1, 1))

    X_scaled       = feat_scaler.transform(X)
    y_price_scaled = price_scaler.transform(y_price.reshape(-1, 1)).flatten()

    X_seq, y_sig_seq, y_prc_seq = create_sequences(
        X_scaled, y_signal, y_price_scaled, lookback
    )

    # seq_dates: preserve actual calendar dates for each sequence end-point
    if "Date" in df_period.columns:
        seq_dates = df_period["Date"].iloc[lookback:].values
    else:
        seq_dates = df_period.index[lookback:]

    # ── Compute split boundaries in sequence-space ────────────
    train_cut = split_index - lookback
    val_size  = max(MIN_VAL_ROWS, int(train_cut * VAL_FRACTION_WITHIN_TRAIN))
    val_start = train_cut - val_size

    # ✅ CHANGE 4: Guard uses MIN_TEST_ROWS (30) instead of hardcoded 20
    # Also add val_size guard so we never train on < 50 rows [web:117]
    min_train_rows = 50
    if (val_start <= 0
            or train_cut <= val_start
            or len(X_seq[train_cut:]) < MIN_TEST_ROWS
            or val_start < min_train_rows):
        return None

    return {
        "X_seq":         X_seq,
        "y_sig_seq":     y_sig_seq,
        "y_prc_seq":     y_prc_seq,
        "train_cut":     train_cut,
        "val_start":     val_start,
        "feat_scaler":   feat_scaler,
        "price_scaler":  price_scaler,
        "raw_df":        df_period,
        "seq_dates":     seq_dates,
    }

In [39]:
# ══════════════════════════════════════════════════════════════
# CELL 7 — DualLSTM Model (updated for 18 features + 1y–5y)
# ══════════════════════════════════════════════════════════════

class DualLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=128, second_hidden=64,
                 dense_size=32, dropout=0.2):
        super().__init__()

        # ── Layer 1 ───────────────────────────────────────────
        self.lstm1 = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.norm1 = nn.LayerNorm(hidden_size)
        self.drop1 = nn.Dropout(dropout)

        # ── Layer 2 ───────────────────────────────────────────
        self.lstm2 = nn.LSTM(hidden_size, second_hidden, batch_first=True)
        self.norm2 = nn.LayerNorm(second_hidden)
        self.drop2 = nn.Dropout(dropout)

        # ── Dense shared trunk ────────────────────────────────
        self.fc1     = nn.Linear(second_hidden, dense_size)
        self.relu    = nn.ReLU()
        self.fc_drop = nn.Dropout(dropout / 2)

        # ✅ CHANGE 1: Add a second dense layer for richer representation
        # With 18 features (up from 11), the shared trunk needs one more
        # transformation step before splitting into two heads [web:106]
        self.fc2     = nn.Linear(dense_size, dense_size)

        # ── Head 1: Signal (BUY/SELL classification) ─────────
        self.signal_head = nn.Linear(dense_size, 1)

        # ── Head 2: Price (next day % return regression) ──────
        # ✅ CHANGE 2: Add a dedicated 1-layer sub-network per head
        # Prevents the signal head from dominating the shared trunk
        # gradients and vice versa — each head now has its own capacity [web:114]
        self.price_fc   = nn.Linear(dense_size, dense_size // 2)
        self.price_head = nn.Linear(dense_size // 2, 1)

        self.signal_fc   = nn.Linear(dense_size, dense_size // 2)
        self.signal_head = nn.Linear(dense_size // 2, 1)

    def forward(self, x):
        # ── LSTM stack ────────────────────────────────────────
        x, _ = self.lstm1(x)
        x     = self.norm1(x)
        x     = self.drop1(x)

        x, _ = self.lstm2(x)
        x     = self.norm2(x)
        x     = self.drop2(x[:, -1, :])   # take last timestep only

        # ── Shared dense trunk ────────────────────────────────
        x = self.relu(self.fc1(x))
        x = self.fc_drop(x)
        x = self.relu(self.fc2(x))         # ✅ second shared layer

        # ── Per-head sub-networks ─────────────────────────────
        sig_out = self.relu(self.signal_fc(x))
        sig_out = self.signal_head(sig_out).squeeze(-1)

        prc_out = self.relu(self.price_fc(x))
        prc_out = self.price_head(prc_out).squeeze(-1)

        return sig_out, prc_out


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ✅ Sanity check — verify model builds correctly with 18 features
_test_input = torch.zeros(2, 10, len(feature_cols)).to(device)
_test_model  = DualLSTM(input_size=len(feature_cols)).to(device)
_s, _p       = _test_model(_test_input)
print(f"Model output shapes — signal: {_s.shape}, price: {_p.shape}")
print(f"✅ DualLSTM built successfully | input_size={len(feature_cols)}")
del _test_input, _test_model

Using device: cuda
Model output shapes — signal: torch.Size([2]), price: torch.Size([2])
✅ DualLSTM built successfully | input_size=18


In [40]:
# ══════════════════════════════════════════════════════════════
# CELL 8 — build_splits_and_loaders, train_one_config, base_search_space
# ALL FIXES: macro-F1 threshold search, per-period ensemble, no vstack
# ══════════════════════════════════════════════════════════════
from sklearn.metrics import accuracy_score, f1_score


def build_splits_and_loaders(config):
    arrays = prepare_period_arrays(period_featured_dfs[config["period"]], config["lookback"])
    if arrays is None:
        return None

    X_seq      = arrays["X_seq"]
    y_sig_seq  = arrays["y_sig_seq"]
    y_prc_seq  = arrays["y_prc_seq"]
    train_cut  = arrays["train_cut"]
    val_start  = arrays["val_start"]

    X_train_np = X_seq[:val_start]
    X_val_np   = X_seq[val_start:train_cut]
    X_test_np  = X_seq[train_cut:]

    y_sig_train_np = y_sig_seq[:val_start]
    y_sig_val_np   = y_sig_seq[val_start:train_cut]
    y_sig_test_np  = y_sig_seq[train_cut:]

    y_prc_train_np = y_prc_seq[:val_start]
    y_prc_val_np   = y_prc_seq[val_start:train_cut]
    y_prc_test_np  = y_prc_seq[train_cut:]

    seq_dates     = arrays["seq_dates"]
    val_dates_np  = np.array(seq_dates[val_start:train_cut])
    test_dates_np = np.array(seq_dates[train_cut:])

    recency_weights_np = np.linspace(0.80, 1.20, len(X_train_np), dtype=np.float32)

    X_train = torch.tensor(X_train_np, dtype=torch.float32)
    X_val   = torch.tensor(X_val_np,   dtype=torch.float32)
    X_test  = torch.tensor(X_test_np,  dtype=torch.float32)

    y_sig_train = torch.tensor(y_sig_train_np, dtype=torch.float32)
    y_sig_val   = torch.tensor(y_sig_val_np,   dtype=torch.float32)
    y_sig_test  = torch.tensor(y_sig_test_np,  dtype=torch.float32)

    y_prc_train = torch.tensor(y_prc_train_np, dtype=torch.float32)
    y_prc_val   = torch.tensor(y_prc_val_np,   dtype=torch.float32)
    y_prc_test  = torch.tensor(y_prc_test_np,  dtype=torch.float32)
    recency_weights = torch.tensor(recency_weights_np, dtype=torch.float32)

    train_ds = TensorDataset(X_train, y_sig_train, y_prc_train, recency_weights)
    train_dl = DataLoader(train_ds, batch_size=config["batch_size"],
                          shuffle=True, drop_last=False)

    return {
        "X_train": X_train, "X_val": X_val, "X_test": X_test,
        "y_sig_train": y_sig_train, "y_sig_val": y_sig_val, "y_sig_test": y_sig_test,
        "y_prc_train": y_prc_train, "y_prc_val": y_prc_val, "y_prc_test": y_prc_test,
        "train_dl":     train_dl,
        "feat_scaler":  arrays["feat_scaler"],
        "price_scaler": arrays["price_scaler"],
        "period_df":    arrays["raw_df"],
        "val_dates":    val_dates_np,
        "test_dates":   test_dates_np,
    }


def weighted_focal_bce_with_logits(logits, targets, pos_weight=None, gamma=1.5):
    bce = nn.functional.binary_cross_entropy_with_logits(
        logits, targets, reduction="none", pos_weight=pos_weight
    )
    probs        = torch.sigmoid(logits)
    p_t          = probs * targets + (1.0 - probs) * (1.0 - targets)
    focal_factor = (1.0 - p_t).pow(gamma)
    return focal_factor * bce


def train_one_config(config):
    bundle = build_splits_and_loaders(config)
    if bundle is None:
        return None

    model = DualLSTM(
        input_size    = len(feature_cols),
        hidden_size   = config["hidden_size"],
        second_hidden = config["second_hidden"],
        dense_size    = config["dense_size"],
        dropout       = config["dropout"]
    ).to(device)

    X_val_dev     = bundle["X_val"].to(device)
    y_sig_val_dev = bundle["y_sig_val"].to(device)
    y_prc_val_dev = bundle["y_prc_val"].to(device)

    pos_count  = float(bundle["y_sig_train"].sum().item())
    neg_count  = float(len(bundle["y_sig_train"]) - pos_count)
    pos_weight = torch.tensor(
        [neg_count / max(pos_count, 1.0)], dtype=torch.float32, device=device
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"]
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer,
        T_0=max(8, config["epochs"] // 4),
        T_mult=1
    )

    best_val_loss  = float("inf")
    best_state     = None
    best_val_probs = None
    wait           = 0

    for epoch in range(1, config["epochs"] + 1):
        model.train()
        for xb, ysb, ypb, wb in bundle["train_dl"]:
            xb, ysb, ypb, wb = (xb.to(device), ysb.to(device),
                                 ypb.to(device), wb.to(device))

            ysb_smooth = ysb * 0.98 + 0.01

            sig_logits, prc_out = model(xb)

            sig_loss_raw = weighted_focal_bce_with_logits(
                sig_logits, ysb_smooth,
                pos_weight=pos_weight, gamma=config["gamma"]
            )
            prc_loss_raw = nn.functional.smooth_l1_loss(prc_out, ypb, reduction="none")

            sig_loss = (sig_loss_raw * wb).mean()
            prc_loss = (prc_loss_raw * wb).mean()
            loss     = config["alpha"] * sig_loss + config["beta"] * prc_loss

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        scheduler.step(epoch - 1 + 1e-8)

        model.eval()
        with torch.no_grad():
            sig_val_logits, prc_val_out = model(X_val_dev)
            val_prob = torch.sigmoid(sig_val_logits).detach().cpu().numpy()

            val_sig_loss = weighted_focal_bce_with_logits(
                sig_val_logits, y_sig_val_dev,
                pos_weight=pos_weight, gamma=config["gamma"]
            ).mean()
            val_prc_loss = nn.functional.smooth_l1_loss(prc_val_out, y_prc_val_dev)
            val_loss     = (config["alpha"] * val_sig_loss +
                            config["beta"]  * val_prc_loss).item()

        if val_loss < best_val_loss:
            best_val_loss  = val_loss
            best_state     = {k: v.detach().cpu().clone()
                              for k, v in model.state_dict().items()}
            best_val_probs = val_prob.copy()
            wait           = 0
        else:
            wait += 1
            if wait >= config["patience"]:
                break

    model.load_state_dict(best_state)

    val_true     = bundle["y_sig_val"].numpy()
    n_thresh     = max(10, int(len(best_val_probs) * 0.20))
    thresh_probs = best_val_probs[-n_thresh:]
    thresh_true  = val_true[-n_thresh:]

    # FIX 1: macro-F1 threshold search, floor raised 0.40 → 0.45
    best_threshold = 0.50
    best_thresh_f1 = -1.0
    for thr in np.arange(0.45, 0.66, 0.01):
        preds = (thresh_probs >= thr).astype(int)
        score = f1_score(thresh_true, preds, average="macro", zero_division=0)
        if score > best_thresh_f1:
            best_thresh_f1 = score
            best_threshold = round(float(thr), 2)

    best_val_acc = accuracy_score(
        val_true, (best_val_probs >= best_threshold).astype(int)
    )
    best_val_f1 = f1_score(
        val_true,
        (best_val_probs >= best_threshold).astype(int),
        average="macro",
        zero_division=0
    )

    return {
        "config":         config,
        "bundle":         bundle,
        "model":          model,
        "best_val_loss":  best_val_loss,
        "best_threshold": best_threshold,
        "best_val_acc":   best_val_acc,
        "best_val_f1":    best_val_f1,
        "best_val_probs": best_val_probs,
    }


# FIX 2: alpha=0.92 / beta=0.08 across all configs
base_search_space = [
    {"lookback": 8,  "hidden_size": 112, "second_hidden": 56,  "dense_size": 28, "dropout": 0.10, "lr": 8e-4, "batch_size": 16, "epochs": 80,  "weight_decay": 1e-5, "alpha": 0.92, "beta": 0.08, "gamma": 1.0,  "patience": 10},
    {"lookback": 10, "hidden_size": 112, "second_hidden": 56,  "dense_size": 28, "dropout": 0.15, "lr": 7e-4, "batch_size": 16, "epochs": 90,  "weight_decay": 1e-5, "alpha": 0.92, "beta": 0.08, "gamma": 1.25, "patience": 12},
    {"lookback": 12, "hidden_size": 144, "second_hidden": 72,  "dense_size": 36, "dropout": 0.18, "lr": 6e-4, "batch_size": 16, "epochs": 100, "weight_decay": 5e-5, "alpha": 0.92, "beta": 0.08, "gamma": 1.25, "patience": 12},
    {"lookback": 15, "hidden_size": 144, "second_hidden": 72,  "dense_size": 36, "dropout": 0.20, "lr": 5e-4, "batch_size": 16, "epochs": 110, "weight_decay": 1e-4, "alpha": 0.92, "beta": 0.08, "gamma": 1.5,  "patience": 14},
    {"lookback": 20, "hidden_size": 160, "second_hidden": 80,  "dense_size": 40, "dropout": 0.25, "lr": 3e-4, "batch_size": 32, "epochs": 120, "weight_decay": 1e-4, "alpha": 0.92, "beta": 0.08, "gamma": 1.5,  "patience": 16},
    {"lookback": 20, "hidden_size": 192, "second_hidden": 96,  "dense_size": 48, "dropout": 0.25, "lr": 2e-4, "batch_size": 32, "epochs": 130, "weight_decay": 3e-4, "alpha": 0.92, "beta": 0.08, "gamma": 1.75, "patience": 18},
    {"lookback": 24, "hidden_size": 192, "second_hidden": 96,  "dense_size": 48, "dropout": 0.28, "lr": 2e-4, "batch_size": 32, "epochs": 130, "weight_decay": 3e-4, "alpha": 0.92, "beta": 0.08, "gamma": 1.75, "patience": 18},
    {"lookback": 30, "hidden_size": 192, "second_hidden": 96,  "dense_size": 48, "dropout": 0.30, "lr": 2e-4, "batch_size": 32, "epochs": 140, "weight_decay": 3e-4, "alpha": 0.92, "beta": 0.08, "gamma": 2.0,  "patience": 18},
]

# Build + run search space
search_space = []
for period in period_featured_dfs.keys():
    for base_cfg in base_search_space:
        cfg = dict(base_cfg)
        cfg["period"] = period
        search_space.append(cfg)

print(f"Total configs to run: {len(search_space)}")

all_runs = []
for idx, config in enumerate(search_space, 1):
    print(f"Running config {idx}/{len(search_space)}: period={config['period']} lookback={config['lookback']}", end="")
    result = train_one_config(config)
    if result is None:
        print(" — skipped: insufficient rows")
        continue
    print(f" — val_acc={result['best_val_acc']:.4f} | macro_f1={result['best_val_f1']:.4f} | thr={result['best_threshold']:.2f} | loss={result['best_val_loss']:.4f}")
    all_runs.append(result)

if not all_runs:
    raise ValueError("No valid training configuration was found.")

# FIX 3: combined score uses macro-F1
for r in all_runs:
    r["combined_score"] = 0.6 * r["best_val_acc"] + 0.4 * r["best_val_f1"]

all_runs = sorted(all_runs, key=lambda r: (-r["combined_score"], r["best_val_loss"]))


# FIX 4: per-period selection — defined here, reused in Cell 9
def select_compatible_top_runs(sorted_runs, max_k=3):
    selected     = []
    seen_periods = set()
    for run in sorted_runs:
        p = run["config"]["period"]
        if p not in seen_periods:
            selected.append(run)
            seen_periods.add(p)
        if len(selected) >= max_k:
            break
    return selected


top_runs = select_compatible_top_runs(all_runs, max_k=3)
top_k    = len(top_runs)

print(f"\nSelected {top_k} ensemble members:")
for i, r in enumerate(top_runs, 1):
    print(f"  {i}. period={r['config']['period']} | val_size={len(r['best_val_probs'])} | acc={r['best_val_acc']:.4f} | macro_f1={r['best_val_f1']:.4f} | score={r['combined_score']:.4f}")

ensemble_weights = np.array(
    [(r["combined_score"] + 1e-6) ** 4 for r in top_runs], dtype=np.float64
)
ensemble_weights /= ensemble_weights.sum()

# FIX 5: weighted avg of per-model thresholds — no vstack crash
ensemble_best_thr = float(np.average(
    [r["best_threshold"] for r in top_runs],
    weights=ensemble_weights
))
ensemble_best_thr = round(max(0.45, min(0.65, ensemble_best_thr)), 2)


# Convenience references
best_run        = top_runs[0]
best_cfg        = best_run["config"]
LOOKBACK        = best_cfg["lookback"]
model           = best_run["model"]
bundle          = best_run["bundle"]
feat_scaler     = bundle["feat_scaler"]
price_scaler    = bundle["price_scaler"]
SELECTED_PERIOD = best_cfg["period"]

X_train     = bundle["X_train"];     X_val     = bundle["X_val"];     X_test     = bundle["X_test"]
y_sig_train = bundle["y_sig_train"];  y_sig_val = bundle["y_sig_val"];  y_sig_test = bundle["y_sig_test"]
y_prc_train = bundle["y_prc_train"];  y_prc_val = bundle["y_prc_val"];  y_prc_test = bundle["y_prc_test"]

ensemble_models        = [r["model"]                  for r in top_runs]
ensemble_configs       = [r["config"]                 for r in top_runs]
ensemble_price_scalers = [r["bundle"]["price_scaler"] for r in top_runs]
ensemble_threshold     = ensemble_best_thr

print(f"\nBest single config : {best_cfg}")
print(f"Ensemble threshold : {ensemble_threshold}  (weighted avg, macro-F1 optimised)")
print(f"Selected period    : {SELECTED_PERIOD}")
print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")


# Save to disk
checkpoint = {
    "ensemble_model_state_dicts": [m.state_dict() for m in ensemble_models],
    "ensemble_configs":           ensemble_configs,
    "ensemble_weights":           ensemble_weights.tolist(),
    "threshold":                  ensemble_best_thr,
    "feature_cols":               feature_cols,
}
torch.save(checkpoint,  "dual_lstm_model.pth")
joblib.dump([r["bundle"]["feat_scaler"]  for r in top_runs], "scaler_features.pkl")
joblib.dump([r["bundle"]["price_scaler"] for r in top_runs], "scaler_price.pkl")
print("✅ Saved Ensemble Models & Scalers to disk!")

Total configs to run: 32
Running config 1/32: period=2y lookback=8 — val_acc=0.6786 | macro_f1=0.6680 | thr=0.52 | loss=0.3039
Running config 2/32: period=2y lookback=10 — val_acc=0.5357 | macro_f1=0.5351 | thr=0.48 | loss=0.3047
Running config 3/32: period=2y lookback=12 — val_acc=0.6429 | macro_f1=0.6410 | thr=0.52 | loss=0.3012
Running config 4/32: period=2y lookback=15 — val_acc=0.5185 | macro_f1=0.5077 | thr=0.51 | loss=0.2542
Running config 5/32: period=2y lookback=20 — val_acc=0.4444 | macro_f1=0.3077 | thr=0.45 | loss=0.2606
Running config 6/32: period=2y lookback=20 — val_acc=0.4444 | macro_f1=0.4414 | thr=0.52 | loss=0.2241
Running config 7/32: period=2y lookback=24 — val_acc=0.5385 | macro_f1=0.5273 | thr=0.53 | loss=0.2258
Running config 8/32: period=2y lookback=30 — val_acc=0.6400 | macro_f1=0.5322 | thr=0.50 | loss=0.1854
Running config 9/32: period=3y lookback=8 — val_acc=0.6102 | macro_f1=0.6084 | thr=0.46 | loss=0.3115
Running config 10/32: period=3y lookback=10 — val_

In [41]:
# ══════════════════════════════════════════════════════════════
# CELL 9 — Train ALL tickers
# ALL FIXES: macro-F1 threshold, per-period ensemble, no vstack
# ══════════════════════════════════════════════════════════════
import os
from sklearn.metrics import accuracy_score, f1_score
os.makedirs("models", exist_ok=True)

# Reuse SPY/VIX from Cell 3 if already downloaded
if "spy_series" not in dir() or not isinstance(spy_series, pd.Series):
    print("Downloading SPY and VIX data...")
    spy_series = yf.download("SPY", period=MAX_DOWNLOAD_PERIOD, interval="1d",
                             progress=False, auto_adjust=True)["Close"].pct_change()
    vix_series = yf.download("^VIX", period=MAX_DOWNLOAD_PERIOD, interval="1d",
                             progress=False, auto_adjust=True)["Close"]
    if isinstance(spy_series, pd.DataFrame): spy_series = spy_series.iloc[:, 0]
    if isinstance(vix_series, pd.DataFrame): vix_series = vix_series.iloc[:, 0]
    spy_series.index = pd.to_datetime(spy_series.index)
    vix_series.index = pd.to_datetime(vix_series.index)
    spy_series.name  = "spy_return"
    vix_series.name  = "vix_level"
    print(f"✅ SPY ({len(spy_series)} rows) and VIX ({len(vix_series)} rows) downloaded.")
else:
    print(f"✅ Reusing SPY ({len(spy_series)} rows) and VIX ({len(vix_series)} rows) from Cell 3.")

all_ticker_results = {}

for TICKER in TICKERS:
    print(f"\n{'='*60}\nTraining: {TICKER}\n{'='*60}")

    # Step 1: Download ticker
    df_full = yf.download(TICKER, period=MAX_DOWNLOAD_PERIOD,
                          interval="1d", progress=False, auto_adjust=True)
    if isinstance(df_full.columns, pd.MultiIndex):
        df_full.columns = [str(c[0]) for c in df_full.columns]
    df_full = df_full.loc[:, ~df_full.columns.duplicated()]
    df_full = df_full.reset_index().sort_values("Date").reset_index(drop=True)
    print(f"Downloaded {len(df_full)} days for {TICKER}")

    # Merge SPY + VIX
    df_full = df_full.set_index("Date")
    df_full = df_full.join(spy_series, how="left").join(vix_series, how="left")
    df_full = df_full.reset_index()
    df_full[["spy_return", "vix_level"]] = df_full[["spy_return", "vix_level"]].ffill()
    print(f"Merged SPY + VIX — NaN: spy={df_full['spy_return'].isna().sum()}, vix={df_full['vix_level'].isna().sum()}")

    # Step 2: Period slicing + class balance check
    period_featured_dfs = {}
    for p in PERIOD_CANDIDATES:
        df_tmp = get_period_df(df_full, p)
        # FIX 6: min rows raised 120 → 250
        if len(df_tmp) >= 250:
            period_featured_dfs[p] = df_tmp

    if not period_featured_dfs:
        print(f"⚠️ Skipping {TICKER} — not enough data after feature engineering")
        continue

    print(f"Usable periods: { {k: len(v) for k, v in period_featured_dfs.items()} }")

    for period, df_p in period_featured_dfs.items():
        buy_pct = df_p["target_signal"].mean() * 100
        status  = "✅" if 40 <= buy_pct <= 65 else "⚠️ IMBALANCED"
        print(f"  {period}: BUY={buy_pct:.1f}% SELL={100-buy_pct:.1f}%  {status}")

    # Step 3: Build search space
    search_space = []
    for period in period_featured_dfs.keys():
        for base_cfg in base_search_space:
            cfg = dict(base_cfg)
            cfg["period"] = period
            search_space.append(cfg)
    print(f"Total configs to run: {len(search_space)}")

    # Step 4: Run all configs
    all_runs = []
    for idx, config in enumerate(search_space, 1):
        print(f"  Config {idx}/{len(search_space)} | period={config['period']} | lookback={config['lookback']}", end="")
        result = train_one_config(config)
        if result is None:
            print(" — skipped (insufficient rows)")
            continue
        print(f" — val_acc={result['best_val_acc']:.4f} macro_f1={result['best_val_f1']:.4f} thr={result['best_threshold']:.2f}")
        all_runs.append(result)

    if not all_runs:
        print(f"❌ No valid runs for {TICKER}, skipping")
        continue

    for r in all_runs:
        r["combined_score"] = 0.6 * r["best_val_acc"] + 0.4 * r["best_val_f1"]
    all_runs = sorted(all_runs, key=lambda r: (-r["combined_score"], r["best_val_loss"]))

    # Step 5: Select top compatible runs (uses function defined in Cell 8)
    top_runs = select_compatible_top_runs(all_runs, max_k=3)
    topk     = len(top_runs)

    print(f"\n  Selected {topk} ensemble members:")
    for i, r in enumerate(top_runs, 1):
        print(f"    {i}. period={r['config']['period']} | val_size={len(r['best_val_probs'])} | acc={r['best_val_acc']:.4f} | macro_f1={r['best_val_f1']:.4f}")

    # Step 6: Ensemble weights + threshold (no vstack)
    ensemble_weights = np.array(
        [(r["combined_score"] + 1e-6) ** 4 for r in top_runs], dtype=np.float64
    )
    ensemble_weights /= ensemble_weights.sum()

    ensemble_best_thr = float(np.average(
        [r["best_threshold"] for r in top_runs],
        weights=ensemble_weights
    ))
    ensemble_best_thr = round(max(0.45, min(0.65, ensemble_best_thr)), 2)

    print(f"✅ {TICKER} | top-{topk} | val_acc={top_runs[0]['best_val_acc']:.4f} | macro_f1={top_runs[0]['best_val_f1']:.4f} | score={top_runs[0]['combined_score']:.4f} | thr={ensemble_best_thr}")

    # Step 7: Save per-ticker
    out_dir = f"models/{TICKER}"
    os.makedirs(out_dir, exist_ok=True)

    checkpoint = {
        "ensemble_model_state_dicts": [r["model"].state_dict() for r in top_runs],
        "ensemble_configs":           [r["config"]             for r in top_runs],
        "ensemble_weights":           ensemble_weights.tolist(),
        "threshold":                  ensemble_best_thr,
        "feature_cols":               feature_cols,
    }
    torch.save(checkpoint,  f"{out_dir}/dual_lstm_model.pth")
    joblib.dump([r["bundle"]["feat_scaler"]  for r in top_runs], f"{out_dir}/scaler_features.pkl")
    joblib.dump([r["bundle"]["price_scaler"] for r in top_runs], f"{out_dir}/scaler_price.pkl")

    all_ticker_results[TICKER] = {
        "val_acc":   top_runs[0]["best_val_acc"],
        "val_f1":    top_runs[0]["best_val_f1"],
        "score":     top_runs[0]["combined_score"],
        "threshold": ensemble_best_thr,
        "topk":      topk,
    }
    print(f"💾 Saved → {out_dir}/\n")

# Final summary
print("\n" + "="*60)
print("TRAINING COMPLETE — Summary:")
print("="*60)
for t, res in all_ticker_results.items():
    print(f"  {t:<6} | acc={res['val_acc']:.4f} | macro_f1={res['val_f1']:.4f} | score={res['score']:.4f} | thr={res['threshold']} | k={res['topk']}")

✅ Reusing SPY (1256 rows) and VIX (1256 rows) from Cell 3.

Training: NVDA
Downloaded 1256 days for NVDA
Merged SPY + VIX — NaN: spy=1, vix=0
Usable periods: {'2y': 252, '3y': 504, '4y': 756, '5y': 1004}
  2y: BUY=54.4% SELL=45.6%  ✅
  3y: BUY=53.2% SELL=46.8%  ✅
  4y: BUY=53.6% SELL=46.4%  ✅
  5y: BUY=53.2% SELL=46.8%  ✅
Total configs to run: 32
  Config 1/32 | period=2y | lookback=8 — val_acc=0.6071 macro_f1=0.6026 thr=0.49
  Config 2/32 | period=2y | lookback=10 — val_acc=0.7500 macro_f1=0.7497 thr=0.51
  Config 3/32 | period=2y | lookback=12 — val_acc=0.7500 macro_f1=0.7333 thr=0.50
  Config 4/32 | period=2y | lookback=15 — val_acc=0.6296 macro_f1=0.6165 thr=0.45
  Config 5/32 | period=2y | lookback=20 — val_acc=0.5185 macro_f1=0.4935 thr=0.49
  Config 6/32 | period=2y | lookback=20 — val_acc=0.5556 macro_f1=0.5235 thr=0.51
  Config 7/32 | period=2y | lookback=24 — val_acc=0.6154 macro_f1=0.6131 thr=0.51
  Config 8/32 | period=2y | lookback=30 — val_acc=0.6800 macro_f1=0.6667 thr=0

In [42]:
# ══════════════════════════════════════════════════════════════
# CELL 10 — Evaluation on Test Set
# ══════════════════════════════════════════════════════════════
from sklearn.metrics import (accuracy_score, classification_report,
                              f1_score, mean_absolute_error, r2_score)

# ── Guard: reload from disk if Cell 8 wasn't run this session ─
_missing = any(v not in dir() for v in [
    "ensemble_models", "ensemble_weights", "ensemble_threshold",
    "top_runs", "best_run", "bundle"
])

if _missing:
    import os

    # ✅ Check per-ticker folder first (Cell 9 output), then root (Cell 8 output)
    _per_ticker_dir = f"models/{TICKER}"
    if os.path.exists(f"{_per_ticker_dir}/dual_lstm_model.pth"):
        _ckpt_path = f"{_per_ticker_dir}/dual_lstm_model.pth"
        _sx_path   = f"{_per_ticker_dir}/scaler_features.pkl"
        _sy_path   = f"{_per_ticker_dir}/scaler_price.pkl"
        print(f"⚠️  Reloading from per-ticker folder: {_per_ticker_dir}/")
    elif os.path.exists("dual_lstm_model.pth"):
        _ckpt_path = "dual_lstm_model.pth"
        _sx_path   = "scaler_features.pkl"
        _sy_path   = "scaler_price.pkl"
        print("⚠️  Reloading from root (Cell 8 output)...")
    else:
        raise FileNotFoundError(
            f"\n❌ No trained model found for '{TICKER}'.\n"
            f"   Expected: {_per_ticker_dir}/dual_lstm_model.pth\n"
            f"   OR:       dual_lstm_model.pth\n\n"
            f"   ✅ Fix: Run Cells 1–8 first to train the model, then re-run Cell 10."
        )

    _ckpt      = torch.load(_ckpt_path, map_location=device, weights_only=False)
    _scalers_X = joblib.load(_sx_path)
    _scalers_y = joblib.load(_sy_path)


    ensemble_weights   = np.array(_ckpt["ensemble_weights"], dtype=np.float64)
    ensemble_threshold = float(_ckpt["threshold"])
    ensemble_configs   = _ckpt["ensemble_configs"]
    feature_cols_ckpt  = _ckpt.get("feature_cols", feature_cols)

    ensemble_models = []
    for i, cfg in enumerate(ensemble_configs):
        m = DualLSTM(
            input_size    = len(feature_cols_ckpt),
            hidden_size   = cfg["hidden_size"],
            second_hidden = cfg["second_hidden"],
            dense_size    = cfg["dense_size"],
            dropout       = cfg["dropout"]
        ).to(device)
        m.load_state_dict(_ckpt["ensemble_model_state_dicts"][i])
        m.eval()
        ensemble_models.append(m)

    print(f"✅ Reloaded {len(ensemble_models)} ensemble models from disk")
    print(f"   threshold={ensemble_threshold} | weights={np.round(ensemble_weights, 4)}")

    # ── Rebuild top_runs / bundle from scratch ─────────────────
    # Download + feature-engineer the ticker data fresh so
    # bundle["X_test"], y_sig_test, y_prc_test etc. are available
    print(f"\nRebuilding data splits for {TICKER} ...")

    df_rebuild = yf.download(TICKER, period=MAX_DOWNLOAD_PERIOD,
                             interval="1d", progress=False, auto_adjust=True)
    if isinstance(df_rebuild.columns, pd.MultiIndex):
        df_rebuild.columns = [str(c[0]) for c in df_rebuild.columns]
    df_rebuild = df_rebuild.loc[:, ~df_rebuild.columns.duplicated()]
    df_rebuild = df_rebuild.reset_index().sort_values("Date").reset_index(drop=True)

    # Re-download SPY/VIX if needed
    if "spy_series" not in dir() or not isinstance(spy_series, pd.Series):
        spy_series = yf.download("SPY", period=MAX_DOWNLOAD_PERIOD, interval="1d",
                                 progress=False, auto_adjust=True)["Close"].pct_change()
        vix_series = yf.download("^VIX", period=MAX_DOWNLOAD_PERIOD, interval="1d",
                                 progress=False, auto_adjust=True)["Close"]
        if isinstance(spy_series, pd.DataFrame): spy_series = spy_series.iloc[:, 0]
        if isinstance(vix_series, pd.DataFrame): vix_series = vix_series.iloc[:, 0]
        spy_series.index = pd.to_datetime(spy_series.index)
        vix_series.index = pd.to_datetime(vix_series.index)
        spy_series.name  = "spy_return"
        vix_series.name  = "vix_level"

    df_rebuild = df_rebuild.set_index("Date")
    df_rebuild = df_rebuild.join(spy_series, how="left").join(vix_series, how="left")
    df_rebuild[["spy_return", "vix_level"]] = df_rebuild[["spy_return", "vix_level"]].ffill()
    df_rebuild = df_rebuild.reset_index()

    # Rebuild period_featured_dfs using the checkpoint's configs
    period_featured_dfs_rebuild = {}
    for cfg in ensemble_configs:
        p = cfg["period"]
        if p not in period_featured_dfs_rebuild:
            df_tmp = get_period_df(df_rebuild, p)
            if len(df_tmp) >= 200:
                period_featured_dfs_rebuild[p] = df_tmp

    # Build top_runs list from checkpoint so bundle is available
    top_runs = []
    for i, cfg in enumerate(ensemble_configs):
        p = cfg["period"]
        if p not in period_featured_dfs_rebuild:
            continue
        arrays = prepare_period_arrays(period_featured_dfs_rebuild[p], cfg["lookback"])
        if arrays is None:
            continue

        val_start = arrays["val_start"]
        train_cut = arrays["train_cut"]
        seq_dates = arrays["seq_dates"]

        def _t(arr, dt=torch.float32):
            return torch.tensor(arr, dtype=dt)

        bundle_r = {
            "X_train":     _t(arrays["X_seq"][:val_start]),
            "X_val":       _t(arrays["X_seq"][val_start:train_cut]),
            "X_test":      _t(arrays["X_seq"][train_cut:]),
            "y_sig_train": _t(arrays["y_sig_seq"][:val_start]),
            "y_sig_val":   _t(arrays["y_sig_seq"][val_start:train_cut]),
            "y_sig_test":  _t(arrays["y_sig_seq"][train_cut:]),
            "y_prc_train": _t(arrays["y_prc_seq"][:val_start]),
            "y_prc_val":   _t(arrays["y_prc_seq"][val_start:train_cut]),
            "y_prc_test":  _t(arrays["y_prc_seq"][train_cut:]),
            "feat_scaler":  _scalers_X[i],
            "price_scaler": _scalers_y[i],
            "period_df":    arrays["raw_df"],
            "val_dates":    np.array(seq_dates[val_start:train_cut]),
            "test_dates":   np.array(seq_dates[train_cut:]),
        }
        top_runs.append({
            "model":          ensemble_models[i],
            "config":         cfg,
            "bundle":         bundle_r,
            "best_val_acc":   0.0,
            "best_val_f1":    0.0,
            "combined_score": ensemble_weights[i],
            "best_threshold": ensemble_threshold,
            "best_val_probs": np.zeros(len(bundle_r["y_sig_val"])),
        })

    if not top_runs:
        raise ValueError("Could not rebuild top_runs from checkpoint — check TICKER and model files.")

    print(f"✅ Rebuilt {len(top_runs)} top_runs from checkpoint")

# ── Convenience references (always set, guard or not) ────────
best_run     = top_runs[0]
bundle       = best_run["bundle"]
price_scaler = bundle["price_scaler"]

# ── Step 1: Ensemble inference on test set ────────────────────
X_test_dev      = bundle["X_test"].to(device)
sig_probs_list  = []
price_pred_list = []

for m in ensemble_models:
    m.eval()
    m.to(device)
    with torch.no_grad():
        sig_logit, prc_out = m(X_test_dev)
    sig_probs_list.append(torch.sigmoid(sig_logit).cpu().numpy())
    price_pred_list.append(prc_out.cpu().numpy())

sig_probs_stack  = np.vstack(sig_probs_list)
price_pred_stack = np.vstack(price_pred_list)

ensemble_sig_probs  = np.average(sig_probs_stack,  axis=0, weights=ensemble_weights)
ensemble_price_pred = np.average(price_pred_stack, axis=0, weights=ensemble_weights)

# ── Step 2: Apply threshold ───────────────────────────────────
sig_pred = (ensemble_sig_probs >= ensemble_threshold).astype(int)

# ── Step 3: Convert % return predictions → dollar prices ──────
price_pred_returns = price_scaler.inverse_transform(
    ensemble_price_pred.reshape(-1, 1)
).flatten()

period_df   = bundle["period_df"]
n_test      = len(bundle["X_test"])
test_df     = period_df.tail(n_test).reset_index(drop=True)
last_closes = test_df["Close"].values

price_pred  = last_closes * (1 + price_pred_returns)

actual_returns       = bundle["y_prc_test"].numpy()
price_actual_returns = price_scaler.inverse_transform(
    actual_returns.reshape(-1, 1)
).flatten()
price_actual = last_closes * (1 + price_actual_returns)

# ── Step 4: Signal metrics ────────────────────────────────────
sig_true = bundle["y_sig_test"].numpy()
test_acc  = accuracy_score(sig_true, sig_pred)
test_f1   = f1_score(sig_true, sig_pred, zero_division=0)

print(f"\n{'='*55}")
print(f"TEST SET RESULTS — {TICKER}")
print(f"{'='*55}")
print(f"Test Accuracy     : {test_acc:.4f}  ({test_acc*100:.1f}%)")
print(f"Test F1 Score     : {test_f1:.4f}")
print(f"Ensemble Threshold: {ensemble_threshold}")
print(f"Test samples      : {len(sig_true)}")
print(f"\nClassification Report:")
print(classification_report(sig_true, sig_pred,
                             target_names=["SELL/HOLD", "BUY"],
                             zero_division=0))

# ── Step 5: Price metrics ─────────────────────────────────────
mae  = mean_absolute_error(price_actual, price_pred)
mape = np.mean(np.abs((price_actual - price_pred) / (price_actual + 1e-9))) * 100
rmse = np.sqrt(np.mean((price_actual - price_pred) ** 2))
r2   = r2_score(price_actual, price_pred)

print(f"Price MAE         : ${mae:.2f}")
print(f"Price RMSE        : ${rmse:.2f}")
print(f"Price MAPE        : {mape:.2f}%")
print(f"Price R²          : {r2:.4f}  {'✅ good' if r2 > 0.85 else '⚠️ low — check lagging'}")

# ── Step 6: Results table ─────────────────────────────────────
results = pd.DataFrame({
    "actual_next_close":    np.round(price_actual, 2),
    "predicted_next_close": np.round(price_pred,   2),
    "error_$":              np.round(price_actual - price_pred, 2),
    "actual_signal":        sig_true.astype(int),
    "predicted_signal":     sig_pred,
    "confidence":           np.round(ensemble_sig_probs, 4),
})

print(f"\nLast 10 test rows:")
print(results.tail(10).to_string(index=False))

# ── Step 7: Confidence collapse diagnostic ────────────────────
conf_first = ensemble_sig_probs[:len(ensemble_sig_probs)//2].mean()
conf_last  = ensemble_sig_probs[len(ensemble_sig_probs)//2:].mean()
trend      = "📉 collapsing" if conf_last < conf_first - 0.05 else "✅ stable"
print(f"\nConfidence trend: first-half avg={conf_first:.4f} → last-half avg={conf_last:.4f}  {trend}")

⚠️  Reloading from per-ticker folder: models/AMD/
✅ Reloaded 3 ensemble models from disk
   threshold=0.5 | weights=[0.4623 0.3141 0.2235]

Rebuilding data splits for AMD ...
✅ Rebuilt 3 top_runs from checkpoint

TEST SET RESULTS — AMD
Test Accuracy     : 0.4510  (45.1%)
Test F1 Score     : 0.6111
Ensemble Threshold: 0.5
Test samples      : 51

Classification Report:
              precision    recall  f1-score   support

   SELL/HOLD       0.17      0.04      0.07        24
         BUY       0.49      0.81      0.61        27

    accuracy                           0.45        51
   macro avg       0.33      0.43      0.34        51
weighted avg       0.34      0.45      0.35        51

Price MAE         : $6.37
Price RMSE        : $9.43
Price MAPE        : 3.04%
Price R²          : 0.7868  ⚠️ low — check lagging

Last 10 test rows:
 actual_next_close  predicted_next_close  error_$  actual_signal  predicted_signal  confidence
            220.18                219.28     0.90          

In [45]:
# ══════════════════════════════════════════════════════════════
# BONUS CELL — Run test evaluation for ALL 10 tickers at once
# ══════════════════════════════════════════════════════════════
summary_results = {}

for TICKER in TICKERS:
    print(f"\n{'='*55}")
    print(f"  {TICKER}")
    print(f"{'='*55}")

    # Reload model
    out_dir    = f"models/{TICKER}"
    checkpoint = torch.load(f"{out_dir}/dual_lstm_model.pth", map_location=device)
    feat_scalers_list  = joblib.load(f"{out_dir}/scaler_features.pkl")
    price_scalers_list = joblib.load(f"{out_dir}/scaler_price.pkl")

    e_configs  = checkpoint["ensemble_configs"]
    e_weights  = np.array(checkpoint["ensemble_weights"])
    e_thr      = checkpoint["threshold"]
    e_fcols    = checkpoint["feature_cols"]

    # Rebuild models
    e_models = []
    for i, cfg in enumerate(e_configs):
        m = DualLSTM(
            input_size    = len(e_fcols),
            hidden_size   = cfg["hidden_size"],
            second_hidden = cfg["second_hidden"],
            dense_size    = cfg["dense_size"],
            dropout       = cfg["dropout"]
        ).to(device)
        m.load_state_dict(checkpoint["ensemble_model_state_dicts"][i])
        m.eval()
        e_models.append(m)

    # Download + engineer features for this ticker
    df_t = yf.download(TICKER, period=MAX_DOWNLOAD_PERIOD, interval="1d",
                       progress=False, auto_adjust=True)
    if isinstance(df_t.columns, pd.MultiIndex):
        df_t.columns = [str(c[0]) for c in df_t.columns]
    df_t = df_t.loc[:, ~df_t.columns.duplicated()].reset_index().sort_values("Date").reset_index(drop=True)
    df_t = df_t.set_index("Date")
    df_t = df_t.join(spy_series, how="left").join(vix_series, how="left").reset_index()
    df_t[["spy_return", "vix_level"]] = df_t[["spy_return", "vix_level"]].ffill()

    # Use best period from checkpoint
    best_period = e_configs[0]["period"]
    best_lb     = e_configs[0]["lookback"]
    df_p        = get_period_df(df_t, best_period)

    arrays = prepare_period_arrays(df_p, best_lb)
    if arrays is None:
        print(f"  ⚠️ Not enough data for {TICKER}, skipping")
        continue

    X_seq     = arrays["X_seq"]
    y_sig_seq = arrays["y_sig_seq"]
    y_prc_seq = arrays["y_prc_seq"]
    train_cut = arrays["train_cut"]

    X_test_np     = X_seq[train_cut:]
    y_sig_test_np = y_sig_seq[train_cut:]
    y_prc_test_np = y_prc_seq[train_cut:]
    X_test_t      = torch.tensor(X_test_np, dtype=torch.float32).to(device)

    if len(X_test_t) == 0:
        print(f"  ⚠️ Empty test set for {TICKER}")
        continue

    # Ensemble inference
    all_sig_probs = []
    all_prc_preds = []
    with torch.no_grad():
        for mi, em in enumerate(e_models):
            s_logits, p_out = em(X_test_t)
            all_sig_probs.append(torch.sigmoid(s_logits).cpu().numpy())
            ps = price_scalers_list[mi]
            p_inv = ps.inverse_transform(p_out.cpu().numpy().reshape(-1, 1)).flatten()
            all_prc_preds.append(p_inv)

    sig_probs = np.average(np.stack(all_sig_probs), axis=0, weights=e_weights)
    prc_preds_pct = np.average(np.stack(all_prc_preds), axis=0, weights=e_weights)

    # Actual prices
    close_vals = df_p["Close"].values
    actual_closes = close_vals[train_cut + best_lb:]
    base_closes   = close_vals[train_cut + best_lb - 1:-1]
    pred_closes   = base_closes * (1 + prc_preds_pct)

    y_pred_sig = (sig_probs >= e_thr).astype(int)
    test_acc   = accuracy_score(y_sig_test_np, y_pred_sig)
    test_f1    = f1_score(y_sig_test_np, y_pred_sig, average="macro", zero_division=0)

    n = min(len(actual_closes), len(pred_closes))
    mae  = np.mean(np.abs(actual_closes[:n] - pred_closes[:n]))
    mape = np.mean(np.abs((actual_closes[:n] - pred_closes[:n]) / actual_closes[:n])) * 100

    print(f"  Test samples : {len(X_test_t)}")
    print(f"  Test Accuracy: {test_acc:.4f}  ({test_acc*100:.1f}%)")
    print(f"  Test Macro F1: {test_f1:.4f}")
    print(f"  Price MAE    : ${mae:.2f}")
    print(f"  Price MAPE   : {mape:.2f}%")
    print(f"  Threshold    : {e_thr}")
    print(classification_report(y_sig_test_np, y_pred_sig,
                                 target_names=["SELL/HOLD", "BUY"], zero_division=0))

    summary_results[TICKER] = {
        "test_acc": test_acc, "test_f1": test_f1,
        "mae": mae, "mape": mape, "n": len(X_test_t)
    }

# ── Final table ───────────────────────────────────────────────
print("\n" + "="*65)
print("ALL TICKERS — TEST SUMMARY")
print("="*65)
print(f"{'Ticker':<8} {'Acc':>7} {'MacroF1':>9} {'MAE':>8} {'MAPE':>7} {'N':>5}")
print("-"*65)
for t, r in summary_results.items():
    print(f"{t:<8} {r['test_acc']:>7.4f} {r['test_f1']:>9.4f} {r['mae']:>7.2f} {r['mape']:>6.2f}% {r['n']:>5}")


  NVDA
  Test samples : 51
  Test Accuracy: 0.5098  (51.0%)
  Test Macro F1: 0.4563
  Price MAE    : $3.31
  Price MAPE   : 1.83%
  Threshold    : 0.5
              precision    recall  f1-score   support

   SELL/HOLD       0.36      0.24      0.29        21
         BUY       0.57      0.70      0.63        30

    accuracy                           0.51        51
   macro avg       0.46      0.47      0.46        51
weighted avg       0.48      0.51      0.49        51


  AMZN
  Test samples : 51
  Test Accuracy: 0.5686  (56.9%)
  Test Macro F1: 0.4806
  Price MAE    : $3.75
  Price MAPE   : 1.74%
  Threshold    : 0.5
              precision    recall  f1-score   support

   SELL/HOLD       0.67      0.17      0.27        24
         BUY       0.56      0.93      0.69        27

    accuracy                           0.57        51
   macro avg       0.61      0.55      0.48        51
weighted avg       0.61      0.57      0.49        51


  AXON
  Test samples : 101
  Test Accura

In [43]:
# ══════════════════════════════════════════════════════════════
# CELL 11 — Latest Prediction (next trading day forecast)
# (updated for 1y–5y + per-model scalers + confidence warning)
# ══════════════════════════════════════════════════════════════

# ── Step 1: Build feature windows per ensemble model ─────────
# ✅ CHANGE 1: Each ensemble model was trained with its OWN scaler
# Using only best_run's scaler for all models causes subtle scale mismatch
# when models were trained on different lookback/period configs [web:126]

sig_probs_all   = []
price_preds_all = []

for i, (m, run) in enumerate(zip(ensemble_models, top_runs)):
    # Use THIS model's own feat_scaler and price_scaler
    run_feat_scaler  = run["bundle"]["feat_scaler"]
    run_price_scaler = run["bundle"]["price_scaler"]
    run_period_df    = run["bundle"]["period_df"]
    run_lookback     = run["config"]["lookback"]

    # Scale features with this model's fitted scaler
    X_all        = run_period_df[feature_cols].values
    X_all_scaled = run_feat_scaler.transform(X_all)

    # Slide window of this model's lookback size
    X_windows = np.array([
        X_all_scaled[j - run_lookback:j]
        for j in range(run_lookback, len(X_all_scaled) + 1)
    ])
    X_tensor = torch.tensor(X_windows, dtype=torch.float32).to(device)

    m.eval()
    m.to(device)
    with torch.no_grad():
        sig_logits, prc_out = m(X_tensor)

    sig_probs_all.append(torch.sigmoid(sig_logits).cpu().numpy())

    # ✅ Inverse transform with this model's OWN price scaler
    prc_np = prc_out.cpu().numpy()
    prc_inv = run_price_scaler.inverse_transform(prc_np.reshape(-1, 1)).flatten()
    price_preds_all.append(prc_inv)

# ── Step 2: Weighted ensemble average ────────────────────────
# Each model may have different window counts due to different lookbacks
# → take only the LAST value from each, then average
last_sig_probs   = np.array([s[-1] for s in sig_probs_all])
last_price_preds = np.array([p[-1] for p in price_preds_all])

prob             = float(np.average(last_sig_probs,   weights=ensemble_weights))
predicted_return = float(np.average(last_price_preds, weights=ensemble_weights))

# ── Step 3: Convert % return → dollar price ──────────────────
# Use best_run's period_df for the last close (most recent data)
period_df   = top_runs[0]["bundle"]["period_df"]
last_close  = float(period_df["Close"].iloc[-1])

predicted_next_close = round(last_close * (1 + predicted_return), 2)
predicted_pct_change = round(predicted_return * 100, 4)

# ── Step 4: Signal decision ───────────────────────────────────
upper_hold = min(0.60, max(0.55, ensemble_threshold + 0.08))
lower_hold = max(0.40, min(0.45, ensemble_threshold - 0.08))

if prob >= upper_hold:
    signal = "BUY"
elif prob <= lower_hold:
    signal = "SELL"
else:
    signal = "HOLD"

# ✅ CHANGE 2: Confidence reliability warning
# If all models disagree heavily, flag it — ensemble std tells you this
sig_std = float(np.std(last_sig_probs))
if sig_std > 0.10:
    confidence_note = f"⚠️  High model disagreement (std={sig_std:.3f}) — treat signal cautiously"
elif abs(prob - ensemble_threshold) < 0.05:
    confidence_note = f"⚠️  Near threshold (diff={abs(prob - ensemble_threshold):.3f}) — borderline signal"
else:
    confidence_note = f"✅ Models agree (std={sig_std:.3f})"

# ── Step 5: Print results ─────────────────────────────────────
print(f"\n{'='*55}")
print(f"NEXT TRADING DAY FORECAST — {TICKER}")
print(f"{'='*55}")
print(f"Signal              : {signal}")
print(f"Confidence          : {round(prob, 4)}")
print(f"Model Agreement     : {confidence_note}")
print(f"Last Close          : ${round(last_close, 2)}")
print(f"Predicted Next Close: ${predicted_next_close}")
print(f"Predicted % Change  : {predicted_pct_change:+.4f}%")
print(f"Threshold Used      : {ensemble_threshold}  (BUY≥{upper_hold:.2f} | SELL≤{lower_hold:.2f})")
print(f"Ensemble size       : {len(ensemble_models)} models")
print(f"{'='*55}")


NEXT TRADING DAY FORECAST — AMD
Signal              : HOLD
Confidence          : 0.4957
Model Agreement     : ⚠️  Near threshold (diff=0.004) — borderline signal
Last Close          : $278.26
Predicted Next Close: $278.89
Predicted % Change  : +0.2279%
Threshold Used      : 0.5  (BUY≥0.58 | SELL≤0.42)
Ensemble size       : 3 models


In [46]:
# ══════════════════════════════════════════════════════════════
# BONUS CELL — Next Trading Day Forecast for ALL 10 Tickers
# ══════════════════════════════════════════════════════════════
import os

all_forecasts = {}

for TICKER in TICKERS:
    print(f"\n{'='*55}")
    print(f"NEXT TRADING DAY FORECAST — {TICKER}")
    print(f"{'='*55}")

    try:
        # ── Step 1: Reload models from disk ──────────────────
        out_dir    = f"models/{TICKER}"
        checkpoint = torch.load(f"{out_dir}/dual_lstm_model.pth", map_location=device)
        feat_scalers_list  = joblib.load(f"{out_dir}/scaler_features.pkl")
        price_scalers_list = joblib.load(f"{out_dir}/scaler_price.pkl")

        e_configs  = checkpoint["ensemble_configs"]
        e_weights  = np.array(checkpoint["ensemble_weights"])
        e_thr      = checkpoint["threshold"]
        e_fcols    = checkpoint["feature_cols"]

        e_models = []
        for i, cfg in enumerate(e_configs):
            m = DualLSTM(
                input_size    = len(e_fcols),
                hidden_size   = cfg["hidden_size"],
                second_hidden = cfg["second_hidden"],
                dense_size    = cfg["dense_size"],
                dropout       = cfg["dropout"]
            ).to(device)
            m.load_state_dict(checkpoint["ensemble_model_state_dicts"][i])
            m.eval()
            e_models.append(m)

        # ── Step 2: Download fresh data for this ticker ───────
        df_t = yf.download(TICKER, period=MAX_DOWNLOAD_PERIOD, interval="1d",
                           progress=False, auto_adjust=True)
        if isinstance(df_t.columns, pd.MultiIndex):
            df_t.columns = [str(c[0]) for c in df_t.columns]
        df_t = df_t.loc[:, ~df_t.columns.duplicated()]
        df_t = df_t.reset_index().sort_values("Date").reset_index(drop=True)
        df_t = df_t.set_index("Date")
        df_t = df_t.join(spy_series, how="left").join(vix_series, how="left").reset_index()
        df_t[["spy_return", "vix_level"]] = df_t[["spy_return", "vix_level"]].ffill()

        # ── Step 3: Build feature windows per ensemble model ──
        sig_probs_all   = []
        price_preds_all = []

        for i, (em, cfg) in enumerate(zip(e_models, e_configs)):
            run_feat_scaler  = feat_scalers_list[i]
            run_price_scaler = price_scalers_list[i]
            run_period       = cfg["period"]
            run_lookback     = cfg["lookback"]

            df_p     = get_period_df(df_t, run_period)
            X_all    = df_p[e_fcols].values
            X_scaled = run_feat_scaler.transform(X_all)

            X_windows = np.array([
                X_scaled[j - run_lookback:j]
                for j in range(run_lookback, len(X_scaled) + 1)
            ])
            X_tensor = torch.tensor(X_windows, dtype=torch.float32).to(device)

            em.eval()
            with torch.no_grad():
                sig_logits, prc_out = em(X_tensor)

            sig_probs_all.append(torch.sigmoid(sig_logits).cpu().numpy())
            prc_inv = run_price_scaler.inverse_transform(
                prc_out.cpu().numpy().reshape(-1, 1)
            ).flatten()
            price_preds_all.append(prc_inv)

        # ── Step 4: Weighted ensemble average ─────────────────
        last_sig_probs   = np.array([s[-1] for s in sig_probs_all])
        last_price_preds = np.array([p[-1] for p in price_preds_all])

        prob             = float(np.average(last_sig_probs,   weights=e_weights))
        predicted_return = float(np.average(last_price_preds, weights=e_weights))

        # ── Step 5: Convert % return → dollar price ───────────
        best_period = e_configs[0]["period"]
        df_best     = get_period_df(df_t, best_period)
        last_close  = float(df_best["Close"].iloc[-1])

        predicted_next_close = round(last_close * (1 + predicted_return), 2)
        predicted_pct_change = round(predicted_return * 100, 4)

        # ── Step 6: Signal decision ───────────────────────────
        upper_hold = min(0.60, max(0.55, e_thr + 0.08))
        lower_hold = max(0.40, min(0.45, e_thr - 0.08))

        if prob >= upper_hold:
            signal = "BUY  🟢"
        elif prob <= lower_hold:
            signal = "SELL 🔴"
        else:
            signal = "HOLD 🟡"

        # ── Step 7: Confidence note ───────────────────────────
        sig_std = float(np.std(last_sig_probs))
        if sig_std > 0.10:
            confidence_note = f"⚠️  High disagreement (std={sig_std:.3f})"
        elif abs(prob - e_thr) < 0.05:
            confidence_note = f"⚠️  Near threshold (diff={abs(prob - e_thr):.3f})"
        else:
            confidence_note = f"✅ Models agree (std={sig_std:.3f})"

        # ── Step 8: Print ─────────────────────────────────────
        print(f"Signal              : {signal}")
        print(f"Confidence          : {round(prob, 4)}")
        print(f"Model Agreement     : {confidence_note}")
        print(f"Last Close          : ${round(last_close, 2)}")
        print(f"Predicted Next Close: ${predicted_next_close}")
        print(f"Predicted % Change  : {predicted_pct_change:+.4f}%")
        print(f"Threshold           : {e_thr}  (BUY≥{upper_hold:.2f} | SELL≤{lower_hold:.2f})")
        print(f"Ensemble size       : {len(e_models)} models")

        all_forecasts[TICKER] = {
            "signal":    signal,
            "prob":      round(prob, 4),
            "last_close":  round(last_close, 2),
            "pred_close":  predicted_next_close,
            "pct_change":  predicted_pct_change,
            "threshold":   e_thr,
            "agreement":   confidence_note,
        }

    except Exception as e:
        print(f"  ❌ Error for {TICKER}: {e}")

# ── Final summary table ───────────────────────────────────────
print("\n\n" + "="*75)
print("ALL TICKERS — NEXT TRADING DAY FORECAST SUMMARY")
print("="*75)
print(f"{'Ticker':<7} {'Signal':<10} {'Conf':>6} {'LastClose':>11} {'PredClose':>11} {'Change%':>9} {'Agreement'}")
print("-"*75)
for t, r in all_forecasts.items():
    print(f"{t:<7} {r['signal']:<10} {r['prob']:>6.4f} ${r['last_close']:>10.2f} ${r['pred_close']:>10.2f} {r['pct_change']:>+8.4f}% {r['agreement']}")


NEXT TRADING DAY FORECAST — NVDA
Signal              : SELL 🔴
Confidence          : 0.4052
Model Agreement     : ✅ Models agree (std=0.058)
Last Close          : $198.35
Predicted Next Close: $198.27
Predicted % Change  : -0.0413%
Threshold           : 0.5  (BUY≥0.58 | SELL≤0.42)
Ensemble size       : 3 models

NEXT TRADING DAY FORECAST — AMZN
Signal              : HOLD 🟡
Confidence          : 0.4857
Model Agreement     : ⚠️  Near threshold (diff=0.014)
Last Close          : $249.7
Predicted Next Close: $249.73
Predicted % Change  : +0.0136%
Threshold           : 0.5  (BUY≥0.58 | SELL≤0.42)
Ensemble size       : 3 models

NEXT TRADING DAY FORECAST — AXON
Signal              : HOLD 🟡
Confidence          : 0.4294
Model Agreement     : ⚠️  High disagreement (std=0.109)
Last Close          : $393.08
Predicted Next Close: $391.36
Predicted % Change  : -0.4373%
Threshold           : 0.49  (BUY≥0.57 | SELL≤0.41)
Ensemble size       : 3 models

NEXT TRADING DAY FORECAST — AAPL
Signal         

In [49]:
# ══════════════════════════════════════════════════════════════
# CELL 12 — FastAPI Backend Inference Wrapper
# (fully synced with Cell 4, Cell 5, Cell 7 updates)
# ══════════════════════════════════════════════════════════════


def get_stock_prediction(ticker_symbol: str) -> dict:
    import yfinance as yf
    import pandas as pd
    import numpy as np
    import torch
    import torch.nn as nn
    import joblib
    import os

    # ── Load checkpoint ───────────────────────────────────────
    model_dir = f"models/{ticker_symbol}"
    ckpt_path = os.path.join(model_dir, "dual_lstm_model.pth")
    sx_path   = os.path.join(model_dir, "scaler_features.pkl")
    sy_path   = os.path.join(model_dir, "scaler_price.pkl")

    if not os.path.exists(ckpt_path):
        raise FileNotFoundError(f"No trained model found for {ticker_symbol} at {ckpt_path}")

    checkpoint       = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    ensemble_configs = checkpoint["ensemble_configs"]
    ensemble_weights = np.array(checkpoint["ensemble_weights"], dtype=np.float64)
    threshold        = float(checkpoint.get("threshold", 0.5))
    scalers_X        = joblib.load(sx_path)
    scalers_y        = joblib.load(sy_path)

    class DualLSTM(nn.Module):
        def __init__(self, input_size, hidden_size=128, second_hidden=64,
                     dense_size=32, dropout=0.2):
            super().__init__()
            self.lstm1       = nn.LSTM(input_size, hidden_size, batch_first=True)
            self.norm1       = nn.LayerNorm(hidden_size)
            self.drop1       = nn.Dropout(dropout)
            self.lstm2       = nn.LSTM(hidden_size, second_hidden, batch_first=True)
            self.norm2       = nn.LayerNorm(second_hidden)
            self.drop2       = nn.Dropout(dropout)
            self.fc1         = nn.Linear(second_hidden, dense_size)
            self.relu        = nn.ReLU()
            self.fc_drop     = nn.Dropout(dropout / 2)
            self.fc2         = nn.Linear(dense_size, dense_size)
            self.signal_fc   = nn.Linear(dense_size, dense_size // 2)
            self.signal_head = nn.Linear(dense_size // 2, 1)
            self.price_fc    = nn.Linear(dense_size, dense_size // 2)
            self.price_head  = nn.Linear(dense_size // 2, 1)

        def forward(self, x):
            x, _ = self.lstm1(x)
            x     = self.norm1(x)
            x     = self.drop1(x)
            x, _ = self.lstm2(x)
            x     = self.norm2(x)
            x     = self.drop2(x[:, -1, :])
            x     = self.relu(self.fc1(x))
            x     = self.fc_drop(x)
            x     = self.relu(self.fc2(x))
            sig_out = self.relu(self.signal_fc(x))
            sig_out = self.signal_head(sig_out).squeeze(-1)
            prc_out = self.relu(self.price_fc(x))
            prc_out = self.price_head(prc_out).squeeze(-1)
            return sig_out, prc_out

    def engineer_features(raw_df: pd.DataFrame) -> pd.DataFrame:
        flat = {}
        src  = raw_df.copy()
        if isinstance(src.columns, pd.MultiIndex):
            src.columns = [str(c[0]) if isinstance(c, tuple) else str(c)
                           for c in src.columns]

        for col in ["Open", "High", "Low", "Close", "Volume"]:
            if col in src.columns:
                extracted = src[col]
                if isinstance(extracted, pd.DataFrame):
                    flat[col] = extracted.iloc[:, 0].to_numpy(dtype=float, na_value=np.nan)
                else:
                    flat[col] = extracted.to_numpy(dtype=float, na_value=np.nan)

        if "Date" in src.columns:
            flat["Date"] = src["Date"].values

        for extra_col in ["spy_return", "vix_level"]:
            if extra_col in src.columns:
                extracted = src[extra_col]
                if isinstance(extracted, pd.DataFrame):
                    flat[extra_col] = extracted.iloc[:, 0].to_numpy(dtype=float, na_value=np.nan)
                else:
                    flat[extra_col] = extracted.to_numpy(dtype=float, na_value=np.nan)

        df     = pd.DataFrame(flat)
        close  = df["Close"]
        volume = df["Volume"]

        df["sma_10"]  = close.rolling(10).mean()
        df["sma_20"]  = close.rolling(20).mean()
        df["sma_50"]  = close.rolling(50).mean()
        df["sma_200"] = close.rolling(200).mean()

        delta    = close.diff()
        gain     = delta.clip(lower=0)
        loss     = -delta.clip(upper=0)
        avg_gain = gain.rolling(14).mean()
        avg_loss = loss.rolling(14).mean()
        rs       = avg_gain / avg_loss.replace(0, np.nan)
        df["rsi_14"] = 100 - (100 / (1 + rs))

        ema12 = close.ewm(span=12, adjust=False).mean()
        ema26 = close.ewm(span=26, adjust=False).mean()
        df["macd"]        = ema12 - ema26
        df["macd_signal"] = df["macd"].ewm(span=9, adjust=False).mean()

        rolling_mean  = close.rolling(20).mean()
        rolling_std   = close.rolling(20).std()
        df["bb_upper"] = rolling_mean + (2 * rolling_std)
        df["bb_lower"] = rolling_mean - (2 * rolling_std)
        df["bb_width"] = (df["bb_upper"] - df["bb_lower"]) / close

        df["return"]         = close.pct_change()
        df["return_3d"]      = close.pct_change(3)
        df["return_5d"]      = close.pct_change(5)
        df["volatility_5d"]  = df["return"].rolling(5).std()
        df["volatility_10d"] = df["return"].rolling(10).std()
        df["volatility_20d"] = df["return"].rolling(20).std()

        df["dist_sma_10"]  = (close - df["sma_10"])  / df["sma_10"]
        df["dist_sma_20"]  = (close - df["sma_20"])  / df["sma_20"]
        df["dist_sma_50"]  = (close - df["sma_50"])  / df["sma_50"]
        df["dist_sma_200"] = (close - df["sma_200"]) / df["sma_200"]

        rolling_52w_high    = close.rolling(252).max()
        rolling_52w_low     = close.rolling(252).min()
        df["dist_52w_high"] = (close - rolling_52w_high) / rolling_52w_high
        df["dist_52w_low"]  = (close - rolling_52w_low)  / rolling_52w_low

        df["volume_sma_20"] = volume.rolling(20).mean()
        df["volume_ratio"]  = volume / df["volume_sma_20"].replace(0, np.nan)

        return df.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)

    feature_cols_local = [
        "rsi_14", "macd", "macd_signal", "bb_width",
        "return_3d", "return_5d", "volatility_5d", "volatility_10d",
        "volatility_20d",
        "dist_sma_10", "dist_sma_20", "dist_sma_50", "dist_sma_200",
        "dist_52w_high", "dist_52w_low",
        "volume_ratio",
        "spy_return", "vix_level",
    ]

    spy_series = yf.download("SPY", period="5y", interval="1d",
                              progress=False, auto_adjust=True)["Close"].pct_change()
    vix_series = yf.download("^VIX", period="5y", interval="1d",
                              progress=False, auto_adjust=True)["Close"]

    if isinstance(spy_series, pd.DataFrame): spy_series = spy_series.iloc[:, 0]
    if isinstance(vix_series, pd.DataFrame): vix_series = vix_series.iloc[:, 0]
    spy_series.index = pd.to_datetime(spy_series.index)
    vix_series.index = pd.to_datetime(vix_series.index)
    spy_series.name  = "spy_return"
    vix_series.name  = "vix_level"

    probs       = []
    prices_raw  = []
    ran_indices = []
    last_close  = None

    for model_idx, cfg in enumerate(ensemble_configs):
        model = DualLSTM(
            input_size    = len(feature_cols_local),
            hidden_size   = cfg["hidden_size"],
            second_hidden = cfg["second_hidden"],
            dense_size    = cfg["dense_size"],
            dropout       = cfg["dropout"]
        )
        model.load_state_dict(
            checkpoint["ensemble_model_state_dicts"][model_idx], strict=True
        )
        model.eval()

        df = yf.download(ticker_symbol, period=cfg["period"], interval="1d",
                         progress=False, auto_adjust=True)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = [col[0] for col in df.columns]
        df = df.loc[:, ~df.columns.duplicated()]
        df = df.reset_index().sort_values("Date").reset_index(drop=True)

        df = df.set_index("Date")
        df = df.join(spy_series, how="left").join(vix_series, how="left")
        df[["spy_return", "vix_level"]] = df[["spy_return", "vix_level"]].ffill()
        df = df.reset_index()

        df = engineer_features(df)

        lookback = cfg["lookback"]
        if len(df) < lookback + 5:
            continue

        last_close = float(df["Close"].iloc[-1])

        X_scaled = scalers_X[model_idx].transform(df[feature_cols_local].values)
        X_input  = torch.tensor(
            X_scaled[-lookback:], dtype=torch.float32
        ).unsqueeze(0)

        with torch.no_grad():
            sig_logit, price_out = model(X_input)

        prob             = float(torch.sigmoid(sig_logit).item())
        predicted_return = float(
            scalers_y[model_idx].inverse_transform([[price_out.item()]])[0][0]
        )

        probs.append(prob)
        prices_raw.append(predicted_return)
        ran_indices.append(model_idx)

    if not probs:
        raise ValueError(f"Not enough processed rows for inference on {ticker_symbol}.")

    active_weights       = ensemble_weights[ran_indices]
    active_weights       = active_weights / active_weights.sum()
    ensemble_prob        = float(np.average(probs,      weights=active_weights))
    ensemble_return_pred = float(np.average(prices_raw, weights=active_weights))

    predicted_next_close = round(last_close * (1 + ensemble_return_pred), 2)
    predicted_pct_change = round(ensemble_return_pred * 100, 4)

    upper_hold = min(0.60, max(0.55, threshold + 0.08))
    lower_hold = max(0.40, min(0.45, threshold - 0.08))

    if ensemble_prob >= upper_hold:
        signal = "BUY"
    elif ensemble_prob <= lower_hold:
        signal = "SELL"
    else:
        signal = "HOLD"

    sig_std = float(np.std(probs)) if len(probs) > 1 else 0.0
    if sig_std > 0.10:
        note = f"High model disagreement (std={sig_std:.3f})"
    elif abs(ensemble_prob - threshold) < 0.05:
        note = f"Near threshold (diff={abs(ensemble_prob - threshold):.3f})"
    else:
        note = "Models agree"

    return {
        "ticker":               ticker_symbol,
        "signal":               signal,
        "confidence":           round(ensemble_prob, 4),
        "confidence_note":      note,
        "predicted_next_close": predicted_next_close,
        "predicted_pct_change": predicted_pct_change,
        "last_close":           round(last_close, 2),
        "threshold_used":       round(threshold, 2),
    }


# ══════════════════════════════════════════════════════════════
# ← LOOP STARTS HERE — fully outside the function (zero indent)
# ══════════════════════════════════════════════════════════════

all_results = {}

for ticker in TICKERS:
    try:
        r = get_stock_prediction(ticker)
        all_results[ticker] = r
        sig_emoji = "🟢" if r["signal"] == "BUY" else ("🔴" if r["signal"] == "SELL" else "🟡")
        print(f"{ticker:<6} | {sig_emoji} {r['signal']:<5} | conf={r['confidence']:.4f} | last=${r['last_close']:>8.2f} | pred=${r['predicted_next_close']:>8.2f} | {r['predicted_pct_change']:+.4f}% | {r['confidence_note']}")
    except Exception as e:
        print(f"{ticker:<6} | ❌ Error: {e}")

NVDA   | 🟡 HOLD  | conf=0.5154 | last=$  200.71 | pred=$  201.94 | +0.6148% | Near threshold (diff=0.015)
AMZN   | 🟡 HOLD  | conf=0.4926 | last=$  253.23 | pred=$  253.27 | +0.0173% | Near threshold (diff=0.007)
AXON   | 🟡 HOLD  | conf=0.4854 | last=$  405.15 | pred=$  404.80 | -0.0876% | Near threshold (diff=0.005)
AAPL   | 🟡 HOLD  | conf=0.4831 | last=$  271.57 | pred=$  271.86 | +0.1061% | Near threshold (diff=0.017)
ORCL   | 🟡 HOLD  | conf=0.4882 | last=$  177.51 | pred=$  177.72 | +0.1183% | Near threshold (diff=0.008)
MSFT   | 🟡 HOLD  | conf=0.4688 | last=$  428.54 | pred=$  427.96 | -0.1365% | Near threshold (diff=0.031)
JPM    | 🟡 HOLD  | conf=0.5273 | last=$  313.42 | pred=$  314.29 | +0.2782% | Near threshold (diff=0.003)
META   | 🟡 HOLD  | conf=0.4881 | last=$  684.42 | pred=$  686.57 | +0.3139% | Models agree
TSLA   | 🔴 SELL  | conf=0.4089 | last=$  407.13 | pred=$  404.25 | -0.7079% | Models agree
AMD    | 🟡 HOLD  | conf=0.4860 | last=$  278.60 | pred=$  279.42 | +0.2950% 